## Connexion à PostgreSQL

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

# ENV SETUP

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

encoded_password = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{encoded_password}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)


## Création des tables

In [ ]:
from sqlalchemy import create_engine, text

sql = """
-- Table des agences
CREATE TABLE IF NOT EXISTS agences (
    agence_id SERIAL PRIMARY KEY,
    agence VARCHAR(100) UNIQUE
);

-- Table des segments clients
CREATE TABLE IF NOT EXISTS segments_client (
    segment_id SERIAL PRIMARY KEY,
    segment_client VARCHAR(50) UNIQUE,
    categorie_risque VARCHAR(50)
);

-- Table des clients
CREATE TABLE IF NOT EXISTS clients (
    client_id VARCHAR PRIMARY KEY,
    segment_client VARCHAR(50),
    score_credit_client NUMERIC(10,2)
);

-- Table des produits
CREATE TABLE IF NOT EXISTS produits (
    produit_id SERIAL PRIMARY KEY,
    produit VARCHAR(100) UNIQUE
);

-- Table des transactions
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id BIGINT PRIMARY KEY,
    client_id VARCHAR,
    date_transaction DATE,
    montant NUMERIC(15,2),
    devise VARCHAR(10),
    taux_change_eur NUMERIC(10,4),
    montant_eur NUMERIC(15,2),
    categorie VARCHAR(100),
    produit VARCHAR(100),
    agence VARCHAR(100),
    type_operation VARCHAR(50),
    statut VARCHAR(50),
    score_credit_client NUMERIC(10,2),
    segment_client VARCHAR(50),
    solde_avant NUMERIC(15,2),
    anomaly_montant NUMERIC(10,2),
    anomaly_score NUMERIC(10,2),
    is_anomaly BOOLEAN,
    annee INT,
    mois INT,
    trimestre INT,
    jour_semaine VARCHAR(20),
    montant_eur_verifie NUMERIC(15,2),
    comparaison VARCHAR(50),
    categorie_risque VARCHAR(50),
    taux_rejet NUMERIC(10,2),
    CONSTRAINT fk_transactions_clients
        FOREIGN KEY (client_id) REFERENCES clients(client_id)
);
"""

with engine.begin() as connection:
    connection.execute(text(sql))

print("✅ Tables créées avec succès")

# Chargement & préparation data

In [ ]:
import pandas as pd

df = pd.read_csv('financecore_clean.csv')


In [ ]:
df.columns

In [ ]:
agences_df = df["agence"].drop_duplicates()
segments_clients_df = df[["segment_client", "categorie_risque"]].drop_duplicates()
clients_df = df[["client_id", "segment_client", "score_credit_client"]].drop_duplicates()
produits_df = df[["produit"]].drop_duplicates()
transactions_df = df[[
    "transaction_id",
    "client_id",
    "date_transaction",
    "montant",
    "devise",
    "taux_change_eur",
    "montant_eur",
    "categorie",
    "produit",
    "agence",
    "type_operation",
    "statut",
    "solde_avant",
    "taux_interet",
    "is_anomaly",
    "annee",
    "mois",
    "trimestre",
    "jour_semaine",
    "montant_eur_verifie",
    "ecart_montant_eur",
    "taux_rejet"
]].drop_duplicates(subset="transaction_id")

# Insertion dimensions

In [ ]:
agences_df = df[["agence"]].drop_duplicates().dropna()

with engine.begin() as conn:
    for agence in agences_df["agence"]:
        conn.execute(text("""
            INSERT INTO agences (agence)
            VALUES (:agence)
            ON CONFLICT (agence) DO NOTHING;
        """), {"agence": agence})

In [ ]:
with engine.begin() as conn:
    for row in segments_clients_df.dropna().itertuples(index=False):
        conn.execute(text("""
            INSERT INTO segments_client (segment_client, categorie_risque)
            VALUES (:segment_client, :categorie_risque)
            ON CONFLICT (segment_client) DO NOTHING;
        """), {
            "segment_client": row.segment_client,
            "categorie_risque": row.categorie_risque
        })

In [ ]:
with engine.begin() as conn:
    for p in produits_df["produit"].dropna().unique():
        conn.execute(text("""
            INSERT INTO produits (produit)
            VALUES (:produit)
            ON CONFLICT (produit) DO NOTHING;
        """), {"produit": p})

In [ ]:
with engine.begin() as conn:
    for row in clients_df.dropna().itertuples(index=False):
        conn.execute(text("""
            INSERT INTO clients (client_id, segment_client, score_credit_client)
            VALUES (:client_id, :segment_client, :score_credit_client)
        """), {
            "client_id": row.client_id,
            "segment_client": row.segment_client,
            "score_credit_client": row.score_credit_client
        })

# Clean + Insert transactions

In [ ]:

df = transactions_df.copy()

# ID
df["transaction_id"] = df["transaction_id"].astype(str)

# DATE (IMPORTANT FIX)
df["date_transaction"] = pd.to_datetime(df["date_transaction"], errors="coerce")

#  NUMERIC FIELDS
num_cols = [
    "client_id", "montant", "taux_change_eur", "montant_eur",
    "solde_avant", "montant_eur_verifie", "taux_rejet"
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

#  DROP INVALID ROWS
df = df.dropna(subset=["transaction_id", "date_transaction"])

In [ ]:
df.to_sql(
    "tmp_transactions",
    engine,
    if_exists="replace",
    index=False
)

In [ ]:
from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO transactions (
            transaction_id, client_id, date_transaction, montant,
            devise, taux_change_eur, montant_eur, categorie,
            produit, agence, type_operation, statut,
            solde_avant, is_anomaly, annee, mois,
            trimestre, jour_semaine, montant_eur_verifie,
            taux_rejet
        )
        SELECT
            transaction_id, client_id, date_transaction, montant,
            devise, taux_change_eur, montant_eur, categorie,
            produit, agence, type_operation, statut,
            solde_avant, is_anomaly, annee, mois,
            trimestre, jour_semaine, montant_eur_verifie,
            taux_rejet
        FROM tmp_transactions
        ON CONFLICT (transaction_id) DO NOTHING;
    """))

    conn.execute(text("DROP TABLE tmp_transactions;"))
    

In [ ]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS clients CASCADE"))
clients_df.to_sql("clients", engine, if_exists="replace", index=False)

# KPI & Queries

In [ ]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT
            agence,
            produit,
            annee,
            mois,
            COUNT(*) AS nombre_transactions,
            SUM(montant_eur) AS total_montant_eur,
            AVG(montant_eur) AS moyenne_montant_eur
        FROM
            transactions
        GROUP BY
            agence,
            produit,
            annee,
            mois
        HAVING
            SUM(montant_eur) > 1000
        ORDER BY
            total_montant_eur DESC
        LIMIT 10;
    '''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

In [ ]:
with engine.begin() as conn:
        result = conn.execute(text('''
        SELECT
            client_id,
            solde_avant
        FROM
            transactions
        WHERE
            solde_avant < (
                SELECT AVG(solde_avant)
                FROM transactions
            )
        ORDER BY
            solde_avant
        LIMIT 10;
'''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

In [ ]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT
            sc.categorie_risque,
            COUNT(*) AS nombre_total_transactions,
            COUNT(*) FILTER (
                WHERE t.statut IN ('Rejeté', 'Refusé', 'Défaut')
            ) AS nombre_defauts,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE t.statut IN ('Rejeté', 'Refusé', 'Défaut')
                ) / NULLIF(COUNT(*), 0),
                2
            ) AS taux_defaut_pourcentage
        FROM transactions t
        JOIN clients c ON t.client_id = c.client_id
        JOIN segments_client sc ON c.segment_client = sc.segment_client
        GROUP BY sc.categorie_risque
        ORDER BY taux_defaut_pourcentage DESC;
    '''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

In [ ]:
with engine.begin() as conn:
    result = conn.execute(text('''
        SELECT
            t.transaction_id,
            t.date_transaction,
            t.montant_eur,
            c.client_id,
            c.segment_client,
            sc.categorie_risque,
            a.agence,
            p.produit,
            t.statut
        FROM transactions t
        INNER JOIN clients c 
            ON t.client_id = c.client_id
        INNER JOIN segments_client sc 
            ON c.segment_client = sc.segment_client
        INNER JOIN agences a 
            ON t.agence = a.agence
        INNER JOIN produits p 
            ON t.produit = p.produit
        ORDER BY t.date_transaction DESC
        LIMIT 10;
    '''))

rows = result.fetchall()

for row in rows:
    print(dict(row._mapping))

# Views

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE OR REPLACE VIEW vw_kpi_transactions AS
        SELECT
            t.agence,
            t.produit,
            t.annee,
            t.mois,
            COUNT(*) AS nombre_transactions,
            SUM(t.montant_eur) AS montant_total_eur,
            AVG(t.montant_eur) AS montant_moyen_eur
        FROM transactions t
        GROUP BY
            t.agence,
            t.produit,
            t.annee,
            t.mois;
    '''))

In [ ]:

query = """
SELECT *
FROM clients c
LEFT JOIN segments_client sc
ON LOWER(TRIM(c.segment_client)) = LOWER(TRIM(sc.segment_client));
"""

df = pd.read_sql(query, engine)

print(df.head())

In [ ]:

sql_drop = "DROP VIEW IF EXISTS vw_kpi_taux_defaut;"

sql_create = """
CREATE VIEW vw_kpi_taux_defaut AS
SELECT
    COALESCE(sc.categorie_risque, 'UNKNOWN') AS categorie_risque,

    COUNT(t.transaction_id) AS nombre_transactions,

    COUNT(t.transaction_id) FILTER (
        WHERE LOWER(TRIM(t.statut)) IN ('rejeté', 'refusé', 'défaut')
    ) AS nombre_defauts,

    ROUND(
        100.0 * COUNT(t.transaction_id) FILTER (
            WHERE LOWER(TRIM(t.statut)) IN ('rejeté', 'refusé', 'défaut')
        ) / NULLIF(COUNT(t.transaction_id), 0),
        2
    ) AS taux_defaut_pourcentage

FROM transactions t
LEFT JOIN clients c
    ON TRIM(t.client_id) = TRIM(c.client_id)
LEFT JOIN segments_client sc
    ON TRIM(c.segment_client) = TRIM(sc.segment_client)
GROUP BY sc.categorie_risque;
"""

with engine.begin() as conn:
    conn.execute(text(sql_drop))
    conn.execute(text(sql_create))

print("✅ VIEW recréée avec succès")

In [ ]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE OR REPLACE VIEW vw_kpi_performance_agence AS
        SELECT
            t.agence,
            COUNT(*) AS nombre_transactions,
            SUM(t.montant_eur) AS montant_total,
            AVG(t.montant_eur) AS montant_moyen,
            COUNT(DISTINCT t.client_id) AS nombre_clients_uniques
        FROM transactions t
        GROUP BY t.agence;
    '''))